# data/train_llama_formatted.csv (학습 데이터 파일 필요)

In [1]:
import os
import torch
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    LlamaForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

!pip install trl # Install trl library
from trl import SFTTrainer

# 1. 환경 설정 및 경로 지정
cache_dir = '/data1/aman/programs/'
os.environ['HF_HOME'] = cache_dir
os.environ["TRANSFORMERS_CACHE"] = cache_dir
output_dir = cache_dir

# 2. 데이터셋 로드 및 전처리 (중복 제거 및 간소화)
dataset_df = pd.read_csv('data/train_llama_formatted.csv')
hf_dataset = Dataset.from_pandas(dataset_df)
hf_dataset = hf_dataset.rename_column("data", "text")
# 불필요한 .map() 연산 제거

# 3. 4-bit 양자화 설정 (모델 로드 전에 반드시 선언)
use_4bit = True
bnb_4bit_compute_dtype = "bfloat16" # Ampere(RTX 3000번대 이상) GPU 권장
bnb_4bit_quant_type = "nf4"
use_nested_quant = False

compute_dtype = getattr(torch, bnb_4bit_compute_dtype)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=use_4bit,
    bnb_4bit_quant_type=bnb_4bit_quant_type,
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=use_nested_quant,
)

# 4. 모델 및 토크나이저 로드 (양자화 설정 적용)
model_id = "NousResearch/Llama-2-7b-chat-hf"
device_map = {"": 0}

# OOM을 방지하기 위해 quantization_config를 반드시 전달해야 합니다.
llama_base_model = LlamaForCausalLM.from_pretrained(
    model_id,
    cache_dir=cache_dir,
    quantization_config=bnb_config,
    device_map=device_map
)

llama_tokenizer = AutoTokenizer.from_pretrained(model_id, cache_dir=cache_dir)
llama_tokenizer.pad_token = llama_tokenizer.eos_token
llama_tokenizer.padding_side = "right"

if llama_base_model.config.vocab_size != len(llama_tokenizer):
    llama_tokenizer.add_tokens([llama_tokenizer.unk_token] * (llama_base_model.config.vocab_size - len(llama_tokenizer)))
    llama_base_model.resize_token_embeddings(len(llama_tokenizer))

# 5. 표준 PEFT(LoRA) 적용 (커스텀 lora_scratch 대신 공식 라이브러리 사용)
llama_base_model = prepare_model_for_kbit_training(llama_base_model) # 4bit 학습 준비

peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.1,
    r=8,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"] # Llama 모델의 어텐션 모듈 타겟팅
)

peft_model = get_peft_model(llama_base_model, peft_config)

# 6. 하드웨어 아키텍처에 따른 자동 dtype 설정 (작성하시려던 로직 완성)
fp16 = False
bf16 = False
if compute_dtype == torch.float16 and use_4bit:
    major, _ = torch.cuda.get_device_capability()
    if major >= 8:
        print("Your GPU supports bfloat16: accelerating training with bf16=True")
        bf16 = True
    else:
        fp16 = True
elif compute_dtype == torch.bfloat16:
    bf16 = True

# 7. 학습 인자(TrainingArguments) 설정
training_arguments = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=10,
    per_device_train_batch_size=4,#  배치가 1이므로 학습이 다소 불안정할 수 있습니다. VRAM이 허용한다면 배치를 늘리거나, VRAM이 부족하다면 1
    gradient_accumulation_steps=8, # 배치 크기가 1이므로 안정성을 위해 4~8 권장
    optim="paged_adamw_8bit",
    save_steps=100, # 0 대신 적절한 체크포인트 저장 주기 설정
    logging_steps=10,
    learning_rate=2e-4,
    weight_decay=0.001,
    fp16=fp16,
    bf16=bf16,
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_ratio=0.03,
    group_by_length=True,
    lr_scheduler_type="cosine",
)

# 8. SFTTrainer 설정 및 학습
trainer = SFTTrainer(
    model=peft_model,
    train_dataset=hf_dataset,
    dataset_text_field="text",
    max_seq_length=2048, # None 대신 명시적으로 설정하여 OOM 방지 (필요 시 1024, 2048 등으로 상향)
    tokenizer=llama_tokenizer,
    args=training_arguments,
    packing=False,
)

# 학습 시작
trainer.train()

# 학습 완료 후 어댑터(LoRA 가중치) 저장
trainer.model.save_pretrained(os.path.join(output_dir, "final_lora_adapter"))

FileNotFoundError: [Errno 2] No such file or directory: 'data/train_llama_formatted.csv'